Imports

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

Environment and paths

In [8]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

BASE = Path.cwd()
DATASET = BASE / "dataset"
TRAIN = DATASET / "train"
TEST = DATASET / "test"

print("Project root:")
print(BASE)

print("\nDataset folder:")
print(DATASET)

print("\nTraining folder:")
print(TRAIN)

print("\nTest folder:")
print(TEST)

Project root:
c:\Users\Rudhrashini\OneDrive\Desktop\ML_Challenge

Dataset folder:
c:\Users\Rudhrashini\OneDrive\Desktop\ML_Challenge\dataset

Training folder:
c:\Users\Rudhrashini\OneDrive\Desktop\ML_Challenge\dataset\train

Test folder:
c:\Users\Rudhrashini\OneDrive\Desktop\ML_Challenge\dataset\test


check expected files

In [9]:
# ============================================================
# CHECK EXPECTED DATASET FILES
# ============================================================

expected_train_files = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv"
]

expected_test_files = [
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv"
]

print("TRAIN FILES ACTUALLY PRESENT:")
for file in TRAIN.iterdir():
    print(" ", file.name)

print("\nTEST FILES ACTUALLY PRESENT:")
for file in TEST.iterdir():
    print(" ", file.name)

print("\nEXPECTED TRAIN FILES:")
for filename in expected_train_files:
    path = TRAIN / filename
    print(filename, "->", path.exists())

print("\nEXPECTED TEST FILES:")
for filename in expected_test_files:
    path = TEST / filename
    print(filename, "->", path.exists())

TRAIN FILES ACTUALLY PRESENT:
  ._train_ground_truth.tsv
  ._train_source1.tsv
  ._train_source2.tsv
  ._train_source3.tsv

TEST FILES ACTUALLY PRESENT:
  ._test_source1.tsv
  ._test_source2.tsv
  ._test_source3.tsv

EXPECTED TRAIN FILES:
train_source1.tsv -> False
train_source2.tsv -> False
train_source3.tsv -> False
train_ground_truth.tsv -> False

EXPECTED TEST FILES:
test_source1.tsv -> False
test_source2.tsv -> False
test_source3.tsv -> False


Check those weird ._ files

In [10]:
# ============================================================
# CHECK ._ FILES
# ============================================================

print("Files beginning with ._:")
for folder in [TRAIN, TEST]:

    print(f"\n{folder}")

    for file in folder.glob("._*"):
        size_mb = file.stat().st_size / (1024 * 1024)

        print(
            f"{file.name:40} "
            f"{size_mb:.4f} MB"
        )

Files beginning with ._:

c:\Users\Rudhrashini\OneDrive\Desktop\ML_Challenge\dataset\train
._train_ground_truth.tsv                 0.0002 MB
._train_source1.tsv                      0.0002 MB
._train_source2.tsv                      0.0002 MB
._train_source3.tsv                      0.0002 MB

c:\Users\Rudhrashini\OneDrive\Desktop\ML_Challenge\dataset\test
._test_source1.tsv                       0.0002 MB
._test_source2.tsv                       0.0002 MB
._test_source3.tsv                       0.0002 MB


Check the structure

In [11]:
print("S1 columns:", list(s1.columns))
print("S2 columns:", list(s2.columns))
print("S3 columns:", list(s3.columns))
print("GT columns:", list(gt.columns))

display(s1.head(5))
display(s2.head(5))
display(s3.head(5))
display(gt.head(5))

NameError: name 's1' is not defined

Basic dataset statistics

In [6]:
print("===== ROW COUNTS =====")
print("Source 1:", len(s1))
print("Source 2:", len(s2))
print("Source 3:", len(s3))
print("Ground Truth:", len(gt))

print("\n===== MEMORY USAGE =====")
print("S1:", round(s1.memory_usage(deep=True).sum() / 1024**2, 2), "MB")
print("S2:", round(s2.memory_usage(deep=True).sum() / 1024**2, 2), "MB")
print("S3:", round(s3.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

===== ROW COUNTS =====
Source 1: 2206821
Source 2: 5034616
Source 3: 5285603
Ground Truth: 2206821

===== MEMORY USAGE =====
S1: 604.43 MB
S2: 1418.8 MB
S3: 1478.47 MB


Missing values

In [7]:
def missing_report(df, name):
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })

    print(f"\n===== {name} =====")
    display(result)

missing_report(s1, "SOURCE 1")
missing_report(s2, "SOURCE 2")
missing_report(s3, "SOURCE 3")


===== SOURCE 1 =====


,missing_count,missing_pct
entity_id,0,0.0
business_name,0,0.0
business_address,0,0.0
country,0,0.0



===== SOURCE 2 =====


,missing_count,missing_pct
entity_id,0,0.00
business_name,2,0.00
business_address,168967,3.36
country,0,0.00



===== SOURCE 3 =====


,missing_count,missing_pct
entity_id,0,0.00
business_name,13,0.00
business_address,175916,3.33
country,0,0.00


Country distribution

In [8]:
for df, name in [(s1, "S1"), (s2, "S2"), (s3, "S3")]:
    print(f"\n===== {name} COUNTRIES =====")
    print(df["country"].value_counts(dropna=False))


===== S1 COUNTRIES =====
country
US       1323633
India     883188
Name: count, dtype: int64

===== S2 COUNTRIES =====
country
US       3016817
India    2017799
Name: count, dtype: int64

===== S3 COUNTRIES =====
country
US       3170056
India    2115547
Name: count, dtype: int64


Duplicate IDs

In [9]:
for df, name in [(s1, "S1"), (s2, "S2"), (s3, "S3")]:
    duplicates = df["entity_id"].duplicated().sum()
    print(f"{name} duplicate entity_ids:", duplicates)

print(
    "Duplicate Source-1 IDs in ground truth:",
    gt["source1_entity_id"].duplicated().sum()
)

S1 duplicate entity_ids: 0
S2 duplicate entity_ids: 0
S3 duplicate entity_ids: 0
Duplicate Source-1 IDs in ground truth: 0


Understand the matches

We need to know whether each S1 business usually has:

0 matches

1 match

2 matches

3+ matches

In [10]:
def count_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0
    return len(str(x).split(","))

gt["num_matches"] = gt["matched_entity_ids"].apply(count_matches)

print(gt["num_matches"].value_counts().sort_index())

print("\nMatch statistics:")
print(gt["num_matches"].describe())

num_matches
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Match statistics:
count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: num_matches, dtype: float64


Singleton analysis

In [11]:
total = len(gt)

singletons = (gt["num_matches"] == 0).sum()
matched = (gt["num_matches"] > 0).sum()

print("Total S1 entities:", total)
print("Singletons:", singletons)
print("Matched:", matched)

print("Singleton %:", round(singletons / total * 100, 2))
print("Matched %:", round(matched / total * 100, 2))

Total S1 entities: 2206821
Singletons: 123247
Matched: 2083574
Singleton %: 5.58
Matched %: 94.42


Look at actual noisy names

In [12]:
display(
    s1[["entity_id", "business_name", "business_address", "country"]]
    .sample(20, random_state=42)
)

,entity_id,business_name,business_address,country
2195840,S1-53356671,Pediatric Medicine PLLC,"4850 20, Otisco, NY",US
1487051,S1-320151505,Fetech National Twin,"19034 Woodburn Road, Woodburn, IN",US
1878277,S1-938947364,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ",US
993966,S1-195839862,Construction Ideaz Papers Private Limited,"Building No.4/606, Prabhul Cottage Karimbalur,...",India
565428,S1-655046555,Penaloza and Bittle First Inc.,"4255 Charleswood Avenue, Memphis, TN",US
1645753,S1-758070915,National Investments LLC,"1641 Virginia Lane, Hueytown, AL",US
1165724,S1-169338169,Reus Idaho Company,"Sanford, ME, 31 Guillemette Street",US
363842,S1-796421498,Creative Projects Limited,"12Thfloor, C, 1204, Roya Oasis, Jankalyan Naga...",India
962445,S1-802535179,Goar Stone LLC,"2105 Wood Ridge Cove, Cedar Park, TX",US
964820,S1-97176033,Lakshmi Consultants Private Limited,"A-301, New Sai Dham Chsl, Ramdev Park Road, Th...",India


In [13]:
display(
    s2[["entity_id", "business_name", "business_address", "country"]]
    .sample(20, random_state=42)
)

,entity_id,business_name,business_address,country
2641297,S2-61703745,Harbor Center,"6309 EVANGELINE TRAIL, AUSTIN, TX",US
1856454,S2-420680068,STEWARD ACE PREMIER,"1029 MAPLE HILL RD, LEBANON, TN",US
371269,S2-190917211,Klapper and Lott,"368 VINE STREET, TOOELE, UT",US
793182,S2-109564077,EAR NOSE & THROAT CARE,"YORK RD, LUTHERVILLE, MD",US
4159519,S2-483058783,Shield Inrfabuild Private Limited,"206 FLAT NO. LG 1 KH NO. 514, 516 SHYAM BHAWAN...",India
2290871,S2-257746699,#holtline,"HOUSTON, 6119 DARLINGHURST DRIVE, TX",US
2479261,S2-476138086,Cancer [Services],NaN,US
606462,S2-420894180,Smt Hermes Private Limited Center,"Delhi, HOUSE NO 138, 2ND FLOOR, BLK C, PKT 2 D...",India
3805434,S2-498993768,AMARDEEP PTER PRIVATE LIMITED,"332NEAR TELCO SERVICE STATION RANGPURI, NEW DE...",India
2960649,S2-54193377,Impex-+-Brothers,"FLAT B 1801, PL-15&17, SECTOR 7, PATEL HERITAG...",India


In [14]:
display(
    s3[["entity_id", "business_name", "business_address", "country"]]
    .sample(20, random_state=42)
)

,entity_id,business_name,business_address,country
726818,S3-239979459,Dr Apex Pvt Ltd Partners,"75 Tulshibaugwale Colonysahakarnagar, Pune, MH",India
1045360,S3-109491785,Shiva Mines Pvt Ltd,"F-17, Sector 22, Noida, Gautam Buddha Nagar, उ...",India
4785624,S3-586835953,Vanguard Métropolitan Sunshine,"3518-B Crestview Ln, Catoosa, Oklahoma",US
2503843,S3-449000694,Vanguard Apoge-Inc,"2150 350, Greencastle, Indiana",US
3149825,S3-397034729,रियल एग्रो प्राइवेट लिमिटेड,"H.no 829 D-225vivek Vihar, Delhi, दिल्ली",India
2749542,S3-553748449,Irinovi,"324 Laramie Ave, Chicago, Illinois",US
3399114,S3-787185173,Jarleon Worldwide Llc,"353 Spruce Pine Road, Abingdon, Maryland",US
1611580,S3-691178620,Sfoasseeg L.L.C.,"1161d Brookside Ave, Evansdale, Iowa",US
3556867,S3-138902486,Brixhalo t/a Creative Enterprises,"6, Madurai Road, Trichy, Tamil Nadu",India
4340188,S3-568450939,Nachiket Solutions Limited,"138 / 6, Kiran Complex Zone - Ii, M. P. Nagar ...",India


Inspect actual positive matches

In [15]:
def show_matches(s1_df, s2_df, s3_df, gt_df, n=10):
    lookup_s1 = s1_df.set_index("entity_id")
    lookup_s2 = s2_df.set_index("entity_id")
    lookup_s3 = s3_df.set_index("entity_id")

    shown = 0

    for _, row in gt_df[gt_df["num_matches"] > 0].head(n).iterrows():
        s1_id = row["source1_entity_id"]
        match_ids = str(row["matched_entity_ids"]).split(",")

        print("\n" + "=" * 80)
        print("SOURCE 1")
        print(lookup_s1.loc[s1_id])

        print("\nMATCHES")

        for mid in match_ids:
            if mid.startswith("S2-"):
                print("\n", lookup_s2.loc[mid])
            elif mid.startswith("S3-"):
                print("\n", lookup_s3.loc[mid])

        shown += 1

show_matches(s1, s2, s3, gt, n=10)


SOURCE 1
business_name           Maure Williams Colombier Inc
business_address    85 Wayne Avenue, Ticonderoga, NY
country                                           US
Name: S1-965667, dtype: object

MATCHES

 business_name       Maure Wilblims Colombier Inc
business_address                             NaN
country                                       US
Name: S2-681193310, dtype: object

 business_name       Maure Williams Colombier
business_address                         NaN
country                                   US
Name: S2-743505751, dtype: object

 business_name                                                Dréxkor
business_address    85 Wanye Avenue, Ticonderoga Townshiip, New York
country                                                           US
Name: S3-775321672, dtype: object

 business_name                       maurewilliamscolombier.com
business_address    Wayne Ave, Ticonderoga Townshiip, New York
country                                                     US
Nam

Check match distribution by country

In [16]:
gt_analysis = gt.merge(
    s1[["entity_id", "country"]],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
)

print(
    gt_analysis.groupby("country")["num_matches"]
    .agg(["count", "mean", "median", "max"])
)

           count      mean  median  max
country                                
India     883188  3.464543     3.0   11
US       1323633  3.459057     3.0   11


Generate ONE clean report

In [17]:
print("=" * 60)
print("AMAZON ML CHALLENGE 2026 - DATASET PROFILE")
print("=" * 60)

print("\nROW COUNTS")
print("S1:", len(s1))
print("S2:", len(s2))
print("S3:", len(s3))
print("GT:", len(gt))

print("\nCOLUMNS")
print("S1:", list(s1.columns))
print("S2:", list(s2.columns))
print("S3:", list(s3.columns))
print("GT:", list(gt.columns))

print("\nCOUNTRIES")
for df, name in [(s1, "S1"), (s2, "S2"), (s3, "S3")]:
    print(f"{name}:")
    print(df["country"].value_counts().to_dict())

print("\nMATCH DISTRIBUTION")
print(gt["num_matches"].value_counts().sort_index().to_dict())

print("\nSINGLETON RATE")
print(f"{singletons}/{total} = {singletons/total*100:.2f}%")

print("\nDUPLICATE IDS")
for df, name in [(s1, "S1"), (s2, "S2"), (s3, "S3")]:
    print(name, df["entity_id"].duplicated().sum())

print("\n" + "=" * 60)

AMAZON ML CHALLENGE 2026 - DATASET PROFILE

ROW COUNTS
S1: 2206821
S2: 5034616
S3: 5285603
GT: 2206821

COLUMNS
S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids', 'num_matches']

COUNTRIES
S1:
{'US': 1323633, 'India': 883188}
S2:
{'US': 3016817, 'India': 2017799}
S3:
{'US': 3170056, 'India': 2115547}

MATCH DISTRIBUTION
{0: 123247, 1: 119157, 2: 375212, 3: 530841, 4: 484115, 5: 321957, 6: 164868, 7: 63968, 8: 18680, 9: 4205, 10: 534, 11: 37}

SINGLETON RATE
123247/2206821 = 5.58%

DUPLICATE IDS
S1 0
S2 0
S3 0



Creating a validation split

In [18]:
VALIDATION_SIZE = 50_000
RANDOM_SEED = 42

val_s1 = s1.sample(
    n=VALIDATION_SIZE,
    random_state=RANDOM_SEED
)

val_ids = set(val_s1["entity_id"])

val_gt = gt[
    gt["source1_entity_id"].isin(val_ids)
].copy()

print("Validation S1:", len(val_s1))
print("Validation GT:", len(val_gt))

Validation S1: 50000
Validation GT: 50000
